In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.enviroment-config

In [0]:
%run ../00-common/04.gold_helpers

In [0]:
target_table=f"{catalog_name}.{gold_schema}.dim_drivers"

In [0]:
from pyspark.sql import functions as F

In [0]:
drivers_df=(spark.table(f"{catalog_name}.{silver_schema}.drivers")
            .filter(F.col("batch_id")==v_batch_id)
)

In [0]:

ref_nationality_region_df=spark.table(f"{catalog_name}.{gold_schema}.ref_nationality_region")


In [0]:
dim_drivers_df=(
    drivers_df
    .join(ref_nationality_region_df,
          drivers_df.nationality==ref_nationality_region_df.nationality,
          how="left"
          )
    .select(
        drivers_df.driver_id,
        drivers_df.driver_name,
        drivers_df.date_of_birth,
        drivers_df.nationality,
        ref_nationality_region_df.region.alias("nationality_region")
        )
       
)



In [0]:
display(dim_drivers_df)

In [0]:
# (
#     dim_drivers_df
#     .write
#     .format("delta")
#     .mode("overwrite")
#     .saveAsTable(target_table)
# )

In [0]:
write_to_gold(
    input_df=dim_drivers_df,
    target_table=target_table,
    merge_condition="t.driver_id=s.driver_id",
    columns_to_update=["driver_name", "date_of_birth", "nationality", "nationality_region"] 
)

In [0]:
display(spark.table(target_table))